In [1]:
import pandas as pd
import numpy as np
import os
import ast
import seaborn as sns
import matplotlib.pyplot as plt
import json
import math

### Some information : 

For each athlete, we get all of its activities. For each activity, if we have access to the splits we use them because it's way more precise. Therefore we will use data that we can get both from the activities and the splits for the analysis.
Data we will use :
1. In df_final : 

- sport_type (Run, Ride, Swim, Workout...)
- start_date
- athlete_id
- activity_id
- distance
- moving_time
- average_heartrate
- average_speed_km_h

2. In df_athletes_stats : (I am manually adding the data for the athletes I don't have their authorization) 

- athlete_id (or Name Firstname to be decided....)
- Total_distance_run
- Total_distance_ride
- Total_distance_swim
- FC Max (?, or only for the athletes in athletes.json ?)
- 5km PB --> + VDOT equivalent
- 10km PB --> + VDOT equivalent
- 21km PB --> + VDOT equivalent
- 42km PB --> + VDOT equivalent
- VDOT Max (corresponding to their best perf, to "evaluate" their level and compare it to other athletes)
- Nb_activities_run
- Nb_activities_ride
- Nb_activities_swim
- Nb_activities_workout
- Nb_activities

3. df_final : (stats by splits for each activity)
- distance
- moving_time
- sport_type
- activity_id
- start_date
- average_heartrate
- athlete_id
- average_speed_km_h

4. df_best_efforts
- distance_activity
- moving_time_activity
- sport_type
- activity_id
- athlete_id
- id
- best_effort_name
- moving_time
- start_date
- distance_best_effort
- pr_rank

In [2]:
folder = "../data/raw"
df = pd.concat([pd.read_csv(os.path.join(folder, f)) for f in os.listdir(folder) if f.endswith(".csv")], ignore_index=True) # Concat toutes les data dans un seul df
pd.set_option("display.max_colwidth", 100)

In [3]:
df.head(3)

,resource_state,athlete,name,distance,moving_time,elapsed_time,total_elevation_gain,type,sport_type,workout_type,...,average_cadence,best_efforts,similar_activities,message,errors,athlete_id,max_watts,weighted_average_watts,private_note,suffer_score
0,3,"{'id': 111468957, 'resource_state': 1}",Sortie vélo le matin,33197.1,4642,4789,405.0,Ride,Ride,NaN,...,NaN,NaN,NaN,NaN,NaN,111468957_Mathis_Durand,NaN,NaN,NaN,NaN
1,3,"{'id': 111468957, 'resource_state': 1}",Sortie vélo dans l'après-midi,44091.6,6030,6081,539.0,Ride,Ride,NaN,...,NaN,NaN,NaN,NaN,NaN,111468957_Mathis_Durand,NaN,NaN,NaN,NaN
2,3,"{'id': 111468957, 'resource_state': 1}",Course à pied le matin,3413.0,1204,1204,42.0,Run,Run,0.0,...,82.2,"[{'id': 59620483376, 'resource_state': 2, 'name': '400m', 'activity': {'id': 14134192706, 'visib...","{'effort_count': 1, 'average_speed': 2.8347176079734218, 'min_average_speed': 2.8347176079734218...",NaN,NaN,111468957_Mathis_Durand,NaN,NaN,NaN,NaN


## Cleaning

In [4]:
# Keep only useful columns
keep_cols = ['athlete', 'distance', 'moving_time', 'total_elevation_gain',
       'sport_type', 'id', 'start_date', 'average_speed', 'max_speed',
       'average_watts', 'average_heartrate', 'max_heartrate', 'splits_metric',
       'best_efforts', 'athlete_id', 'max_watts', 'weighted_average_watts']
df = df[keep_cols]
df.head(3)

,athlete,distance,moving_time,total_elevation_gain,sport_type,id,start_date,average_speed,max_speed,average_watts,average_heartrate,max_heartrate,splits_metric,best_efforts,athlete_id,max_watts,weighted_average_watts
0,"{'id': 111468957, 'resource_state': 1}",33197.1,4642,405.0,Ride,14201150739,2025-04-17T07:21:57Z,7.151,15.240,121.0,135.0,161.0,"[{'distance': 1012.7, 'elapsed_time': 186, 'elevation_difference': 21.6, 'moving_time': 183, 'sp...",NaN,111468957_Mathis_Durand,NaN,NaN
1,"{'id': 111468957, 'resource_state': 1}",44091.6,6030,539.0,Ride,14145949339,2025-04-11T11:56:10Z,7.312,15.940,129.0,135.3,165.0,"[{'distance': 1007.8, 'elapsed_time': 164, 'elevation_difference': 17.2, 'moving_time': 161, 'sp...",NaN,111468957_Mathis_Durand,NaN,NaN
2,"{'id': 111468957, 'resource_state': 1}",3413.0,1204,42.0,Run,14134192706,2025-04-10T07:53:16Z,2.835,3.383,NaN,154.2,168.0,"[{'distance': 1001.8, 'elapsed_time': 354, 'elevation_difference': 1.8, 'moving_time': 354, 'spl...","[{'id': 59620483376, 'resource_state': 2, 'name': '400m', 'activity': {'id': 14134192706, 'visib...",111468957_Mathis_Durand,NaN,NaN


In [5]:
# Creating the athlete_id column from the athlete column
df['athlete'] = df['athlete'].apply(ast.literal_eval)
df['athlete_id'] = df['athlete'].apply(lambda x: x['id'] if isinstance(x, dict) else None)
df = df.drop(columns=['athlete'])
df = df.rename(columns={'id': 'activity_id'})

# Speed formatting
df['moving_time'] = df['moving_time'] / 60  # minutes
df['average_speed_km_h'] = df['average_speed'] * 3.6
df['max_speed_km_h_activity'] = df['max_speed'] * 3.6
df = df.drop(columns=["average_speed", 'max_speed'])

df['start_date'] = pd.to_datetime(df['start_date'])

# Renaming some columns to avoid conflicts in splits
df = df.rename(columns={
    'distance': 'distance_activity',
    'moving_time': 'moving_time_activity',
    'average_speed_km_h': 'average_speed_km_h_activity',
    'average_heartrate': 'average_heartrate_activity',
    'total_elevation_gain': 'elevation_gain_activity',
    'max_heartrate': 'max_heartrate_activity',
    'average_watts': 'average_watts_activity',
    'max_watts': 'max_watts_activity',
    'weighted_average_watts': 'weighted_average_watts_activity'
})

# Sort for cumulative distance computation
df = df.sort_values(by=['athlete_id', 'start_date'])

# Compute cumulative distance by sport
def cumulative_distance(df, sport):
    mask = df['sport_type'] == sport
    return (
        df.groupby('athlete_id')['distance_activity']
        .transform(lambda x: x.where(mask).cumsum())
    )

df['cumulative_distance_run'] = cumulative_distance(df, 'Run')
df['cumulative_distance_ride'] = cumulative_distance(df, 'Ride')
df['cumulative_distance_swim'] = cumulative_distance(df, 'Swim')

# splits_metric
df['splits_metric'] = df['splits_metric'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df_splits_metric = df.explode("splits_metric").reset_index(drop=True)
df_splits_metric = pd.concat([df_splits_metric.drop(columns=["splits_metric"]), df_splits_metric["splits_metric"].apply(pd.Series)], axis=1)
df = df.drop(columns=["splits_metric"])

# best_efforts
df['best_efforts'] = df['best_efforts'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df_efforts = df[df['best_efforts'].apply(lambda x: isinstance(x, list) and len(x) > 0)].copy() # Delete rows without best_efforts
df_best_efforts = df_efforts.explode("best_efforts").reset_index(drop=True) # Explode and keep useful columns
df_best_efforts = pd.concat(
    [df_best_efforts.drop(columns=["best_efforts", 'start_date']), df_best_efforts["best_efforts"].apply(pd.Series)],
    axis=1
)
# Rename/add useful columns
df_best_efforts['elapsed_time_best_effort_min'] = df_best_efforts['elapsed_time'] / 60
df_best_efforts = df_best_efforts.rename(columns={
    'name': 'best_effort_name',
    'elapsed_time': 'elapsed_time_best_effort',
    'distance': 'distance_best_effort'
})

# df_splits_metric
df_splits_metric['start_date'] = pd.to_datetime(df_splits_metric['start_date'])
df_splits_metric['average_speed_km_h'] = df_splits_metric['average_speed'] * 3.6
df_splits_metric = df_splits_metric.rename(columns={
    'distance': 'distance_split',
    'moving_time': 'moving_time_split',
    'average_speed': 'average_speed_split',
    'average_speed_km_h': 'average_speed_km_h_split',
    'average_heartrate': 'average_heartrate_split'
})
df_splits_metric = df_splits_metric.drop(columns=[
    'elapsed_time', 'elevation_difference', 0, 'average_grade_adjusted_speed',
    'cumulative_distance_swim', 'cumulative_distance_ride', 'cumulative_distance_run',
    'pace_zone', 'split'
], errors='ignore')

# delete 'best_efforts' and 'splits_metric' in other dataframes (more convenient)
for col in ['splits_metric', 'best_efforts']:
    for d in [df, df_splits_metric, df_best_efforts]:
        if col in d.columns:
            d.drop(columns=[col], inplace=True)

df_best_efforts = df_best_efforts.dropna(axis=0, subset = ['pr_rank'])  # Delete rows without pr_rank

keep_cols_best_efforts = ['distance_activity', 'moving_time_activity', 'sport_type',
       'activity_id', 'athlete_id', 'id', 'best_effort_name', 'moving_time',
       'start_date', 'distance_best_effort', 'pr_rank']
df_best_efforts = df_best_efforts[keep_cols_best_efforts]
df_best_efforts['start_date'] = pd.to_datetime(df_best_efforts['start_date'])
df_best_efforts['moving_time'] = df_best_efforts['moving_time'] / 60  # minutes
df_best_efforts = df_best_efforts.drop(            # Only keep records on 5, 10, 21 and 42km
    df_best_efforts[
        (df_best_efforts['best_effort_name'] == '2 mile') |
        (df_best_efforts['best_effort_name'] == '1/2 mile') |
        (df_best_efforts['best_effort_name'] == '10 mile') |
        (df_best_efforts['best_effort_name'] == '1K') |
        (df_best_efforts['best_effort_name'] == '400m') |
        (df_best_efforts['best_effort_name'] == '15K') |
        (df_best_efforts['best_effort_name'] == '1 mile')
    ].index
)

In [6]:
# Step 1 : no splits activities
df_activity_no_splits = df_splits_metric[df_splits_metric['distance_split'].isnull()].copy()

# Step 2 : splits activities
df_activity_only_splits = df_splits_metric[df_splits_metric['distance_split'].notnull()].copy()

# Step 3 : We get rid of all columns related to splits in df_activity_no_splits and the same for columns related to activities in df_activity_only_splits
df_activity_only_splits = df_activity_only_splits.drop(columns=[col for col in df_activity_only_splits.columns if 'activity' in col and col != 'activity_id'], errors='ignore') # We keep 'activity_id' for the fusion
df_activity_no_splits = df_activity_no_splits.drop(columns=[col for col in df_activity_no_splits.columns if 'split' in col], errors='ignore')

# Step 4 : We rename columns to have the same in both dataframes
df_activity_only_splits = df_activity_only_splits.rename(columns={
    'distance_split': 'distance',
    'moving_time_split': 'moving_time',
    'average_speed_km_h_split': 'average_speed_km_h',
    'average_heartrate_split': 'average_heartrate'
})
df_activity_only_splits = df_activity_only_splits.drop(columns=['average_speed_split'])

df_activity_no_splits = df_activity_no_splits.rename(columns={
    'distance_activity': 'distance',
    'moving_time_activity': 'moving_time',
    'average_speed_km_h_activity': 'average_speed_km_h',
    'average_heartrate_activity': 'average_heartrate'
})
df_activity_no_splits = df_activity_no_splits.drop(columns=[col for col in df_activity_no_splits.columns if col not in df_activity_only_splits.columns], errors='ignore') # We keep 'activity_id' for the fusion

# Step 5 : We merge the two dataFrames to have our final df
df_final = pd.concat([df_activity_no_splits, df_activity_only_splits], ignore_index=True)
df_final.dropna(axis=0, subset=['average_heartrate'], inplace=True) # Get rid of rows without average_heartrate (important for our analysis)
df_final.shape

(40484, 8)

In [7]:
df_final.head(3)

,distance,moving_time,sport_type,activity_id,start_date,average_heartrate,athlete_id,average_speed_km_h
0,0.0,0.133333,Run,2804607620,2019-04-13 13:25:12+00:00,125.2,44410997,0.0000
1,0.0,0.000000,Swim,3918089530,2020-08-15 14:28:08+00:00,93.3,44410997,0.0000
2,189.6,4.100000,Workout,4543820130,2020-12-29 15:40:47+00:00,89.1,44410997,2.7756


## Features

In [8]:
# We create a DataFrame to store athletes stats
df_athletes_stats = pd.DataFrame(columns=['athlete_id', 'Total_distance_run', 'Total_distance_ride', 'Total_distance_swim', 'FC Max', '5km PB', '10km PB', '21km PB',
                                        '42km PB', 'Nb_activities', 'Nb_activities_run', 'Nb_activities_ride', 'Nb_activities_swim'])
df_athletes_stats['Nb_activities_run'] = df_final[df_final['sport_type']=='Run'].groupby('athlete_id')['activity_id'].nunique() # We count the number of activities per athlete for each sport
df_athletes_stats['Nb_activities_ride'] = df_final[df_final['sport_type']=='Ride'].groupby('athlete_id')['activity_id'].nunique()
df_athletes_stats['Nb_activities_swim'] = df_final[df_final['sport_type']=='Swim'].groupby('athlete_id')['activity_id'].nunique()
df_athletes_stats['Nb_activities_workout'] = df_final[df_final['sport_type']=='Workout'].groupby('athlete_id')['activity_id'].nunique()

df_athletes_stats['athlete_id'] = df_final['athlete_id'].unique() # We add athlete IDs

# We look at the total distances covered by each athlete for each sport
total_distance_run = df_final[df_final['sport_type']=='Run'].groupby('athlete_id')['distance'].sum()
total_distance_ride = df_final[df_final['sport_type']=='Ride'].groupby('athlete_id')['distance'].sum()
total_distance_swim = df_final[df_final['sport_type']=='Swim'].groupby('athlete_id')['distance'].sum()
df_athletes_stats['Total_distance_run'] = df_athletes_stats['athlete_id'].map(total_distance_run)
df_athletes_stats['Total_distance_ride'] = df_athletes_stats['athlete_id'].map(total_distance_ride)
df_athletes_stats['Total_distance_swim'] = df_athletes_stats['athlete_id'].map(total_distance_swim)

# We look at the max HR of each athlete (we take the average of the 10 highest max HR, without counting those >210 bpm)
fc_valides = df[df["max_heartrate_activity"] < 210].groupby('athlete_id')["max_heartrate_activity"].apply(lambda x: x.nlargest(10).mean())
df_athletes_stats['FC Max'] = df_athletes_stats['athlete_id'].map(fc_valides)

def extract_best_effort_time(df_best_efforts, distance_km):
    return (
        df_best_efforts[df_best_efforts['distance_best_effort'].between((distance_km - 0.2)*1000, (distance_km + 0.2)*1000)]
        .groupby('athlete_id')['moving_time']
        .min()
        .round(2)
    )

df_athletes_stats['5km PB'] = df_athletes_stats['athlete_id'].map(extract_best_effort_time(df_best_efforts, 5))
df_athletes_stats['10km PB'] = df_athletes_stats['athlete_id'].map(extract_best_effort_time(df_best_efforts, 10))
df_athletes_stats['21km PB'] = df_athletes_stats['athlete_id'].map(extract_best_effort_time(df_best_efforts, 21.1))
df_athletes_stats['42km PB'] = df_athletes_stats['athlete_id'].map(extract_best_effort_time(df_best_efforts, 42.2))

In [11]:
# Manually added stats
manual_stats = pd.read_csv("../data/manual_stats.csv")

# Reorganize columns according to the columns present at load time
common_columns = [col for col in df_athletes_stats.columns if col in manual_stats.columns]
manual_stats = manual_stats[common_columns]

# We concatenate the two DataFrames (pandas will fill missing columns with NaN)
df_athletes_stats = pd.concat([df_athletes_stats, manual_stats], ignore_index=True)

df_athletes_stats = df_athletes_stats.drop_duplicates(subset='athlete_id', keep='last') # Getting rid of duplicates (just in case)

# Then preprocessing of the whole dataframe
df_athletes_stats['Nb_activities_run'] = df_athletes_stats['Nb_activities_run'].fillna(0)
df_athletes_stats['Nb_activities_ride'] = df_athletes_stats['Nb_activities_ride'].fillna(0)
df_athletes_stats['Nb_activities_swim'] = df_athletes_stats['Nb_activities_swim'].fillna(0)
df_athletes_stats['Nb_activities_workout'] = df_athletes_stats['Nb_activities_workout'].fillna(0)
df_athletes_stats['Total_distance_run'] = df_athletes_stats['Total_distance_run'].fillna(0)
df_athletes_stats['Total_distance_ride'] = df_athletes_stats['Total_distance_ride'].fillna(0)
df_athletes_stats['Total_distance_swim'] = df_athletes_stats['Total_distance_swim'].fillna(0)
df_athletes_stats['Nb_activities'] = df_athletes_stats['Nb_activities_run'] + df_athletes_stats['Nb_activities_ride'] + df_athletes_stats['Nb_activities_swim'] + df_athletes_stats['Nb_activities_workout']


# Transforming each PB into its equivalent VDOT score (approximation) to better compare performances (Daniels, J. (2013). Daniels’ Running Formula (3rd ed.). Human Kinetics)
def calculate_vdot(distance_m, time_sec):
    time_min = time_sec / 60
    if time_min == 0:
        return None

    velocity = distance_m / time_min  # m/min
    vo2 = -4.6 + 0.182258 * velocity + 0.000104 * velocity**2

    percent_max = 0.8 + 0.1894393 * math.exp(-0.012778 * time_min) + \
                  0.2989558 * math.exp(-0.1932605 * time_min)

    vdot = vo2 / percent_max
    return round(vdot, 2)

for dist, colname in [(5000, '5km PB'), (10000, '10km PB'), (21100, '21km PB'), (42200, '42km PB')]:
    vdot_col = f'VDOT_{colname.split()[0]}'
    df_athletes_stats[vdot_col] = df_athletes_stats.apply(
        lambda row: calculate_vdot(dist, row[colname]*60) if pd.notna(row[colname]) else None, axis=1)

# And then we create the VDOT max variable to "rate" the athletes and standardize performances
df_athletes_stats['VDOT_max'] = df_athletes_stats[['VDOT_5km', 'VDOT_10km', 'VDOT_21km', 'VDOT_42km']].max(axis=1)

In [ ]:
    # We create a 'cv_speed' variable which corresponds to the Coefficient of Variation of speed for each activity (to better differentiate interval training sessions for example: if cv is high, it means
    # that the athlete varied his speed a lot, so he did interval training, if cv is low, he did a steady pace run) 
    # We also create a zone variable which corresponds to the heart rate zone (Z1, Z2, Z3, Z4) for each activity (I used the model from Leif Inge Tjelta for the zones, except that I combined Z4 and Z5)
    # All of these features will be useful for clustering later
def process_splits(splits_df, fc_max_dict):
    # Takes as input:
    # - splits_df: a DataFrame containing the splits of activities (with columns 'activity_id', 'athlete_id', 'average_heartrate', 'average_speed_km_h')
    # - fc_max_dict: a dictionary {athlete_id: fc_max}
    # Returns:
    # - A DataFrame with for each activity:
    #     - cv_speed: coefficient of variation of speed
    #     - pct_Z1 to pct_Z4: percentage of time spent in each HR zone
     splits_df = splits_df.copy()

    # We define the HR zone thresholds as a percentage of max HR
     def get_hr_zone(hr, fc_max):
        if pd.isna(hr) or pd.isna(fc_max):
            return None
        ratio = hr / fc_max
        if ratio < 0.82:
            return "Z1"
        elif ratio < 0.92:
            return "Z2"
        elif ratio < 0.97:
            return "Z3"
        else:
            return "Z4"

    # Apply the heart rate zone to each row
     splits_df["fc_max"] = splits_df["athlete_id"].map(fc_max_dict)
     splits_df["hr_zone"] = splits_df.apply(lambda row: get_hr_zone(row["average_heartrate"], row["fc_max"]), axis=1)

     results = []
     for activity_id, group in splits_df.groupby("activity_id"):
        # Coefficient of variation of speed
        cv_speed = group["average_speed_km_h"].std() / group["average_speed_km_h"].mean()

        # Percentage by heart rate zone
        zone_pct = group["hr_zone"].value_counts(normalize=True).reindex(["Z1", "Z2", "Z3", "Z4"], fill_value=0)
        zone_pct.index = [f"pct_{z}" for z in zone_pct.index]

        res = {"activity_id": activity_id, "cv_speed": cv_speed}
        res.update(zone_pct.to_dict())
        results.append(res)

     return pd.DataFrame(results)


# Create a dictionary to store the max HR of each athlete
fc_max_dict = df_athletes_stats.set_index("athlete_id")["FC Max"].to_dict()
# Applying the function on the splits/activities df
df_processed_splits = process_splits(df_final, fc_max_dict)
# And then we merge
df_final = df_final.merge(df_processed_splits, on="activity_id", how="left")
df_final['cv_speed'] = df_final['cv_speed'].fillna(0) # We suppose that whole activities (without splits) have a cv of 0, so at a constant pace

/tmp/ipykernel_889/1135501917.py:55: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_final['cv_speed'].fillna(0, inplace=True) # We suppose that whole activities (without splits) have a cv of 0, so at a constant pace


In [13]:
df_final.tail(5)
# df_final.shape

,distance,moving_time,sport_type,activity_id,start_date,average_heartrate,athlete_id,average_speed_km_h,cv_speed,pct_Z1,pct_Z2,pct_Z3,pct_Z4
40479,1001.8,288.0,Run,15367826279,2025-08-06 17:04:20+00:00,173.305556,146587351,12.528,0.028496,0.1,0.8,0.1,0.0
40480,999.7,294.0,Run,15367826279,2025-08-06 17:04:20+00:00,179.411565,146587351,12.240,0.028496,0.1,0.8,0.1,0.0
40481,999.1,292.0,Run,15367826279,2025-08-06 17:04:20+00:00,184.397260,146587351,12.312,0.028496,0.1,0.8,0.1,0.0
40482,999.2,300.0,Run,15367826279,2025-08-06 17:04:20+00:00,181.453333,146587351,11.988,0.028496,0.1,0.8,0.1,0.0
40483,853.6,248.0,Run,15367826279,2025-08-06 17:04:20+00:00,188.487903,146587351,12.384,0.028496,0.1,0.8,0.1,0.0


In [14]:
df_final_by_activity = df_final[['athlete_id', 'activity_id', 'start_date', 'cv_speed', 'pct_Z1', 'pct_Z2', 'pct_Z3', 'pct_Z4', 'sport_type']].copy() # We only keep columns useful to group by activities
df_final_by_activity['distance'] = df_final.groupby('activity_id')['distance'].transform('sum')
df_final_by_activity['moving_time'] = df_final.groupby('activity_id')['moving_time'].transform('sum')
df_final_by_activity = df_final_by_activity.drop_duplicates(subset='activity_id') # We get rid of duplicates (because several splits for the same activity)

# We add the "Intensity" and "Training Load" variables (to be adjusted)
# Another option: training_load = intensity * moving_time * (1 + average_speed_km_h / 20)
# something like that to take speed into account
# OR internal load with HR, cv_speed... and external load with distance/moving_time, speed...
df_final_by_activity['intensity'] = (df_final_by_activity["pct_Z1"] * 1 + df_final_by_activity["pct_Z2"] * 1.5 + df_final_by_activity["pct_Z3"] * 2.25 + df_final_by_activity["pct_Z4"] * 3) #+ df_final_by_activity['cv_speed'] * 2
df_final_by_activity['training_load'] = df_final_by_activity['intensity'] * df_final_by_activity['distance']

df_final_by_activity.tail()

,athlete_id,activity_id,start_date,cv_speed,pct_Z1,pct_Z2,pct_Z3,pct_Z4,sport_type,distance,moving_time,intensity,training_load
40448,146587351,15188081520,2025-07-21 15:18:34+00:00,0.332203,0.875,0.125,0.0,0.0,Run,7050.8,2701.0,1.0625,7491.475
40456,146587351,15233239475,2025-07-25 15:49:43+00:00,0.022007,1.000,0.000,0.0,0.0,Run,7122.9,2362.0,1.0000,7122.900
40464,146587351,15341622308,2025-08-04 14:07:06+00:00,0.032626,1.000,0.000,0.0,0.0,Run,7448.0,2639.0,1.0000,7448.000
40472,146587351,15352406696,2025-08-05 12:30:58+00:00,1.414214,1.000,0.000,0.0,0.0,Swim,1000.0,1267.0,1.0000,1000.000
40474,146587351,15367826279,2025-08-06 17:04:20+00:00,0.028496,0.100,0.800,0.1,0.0,Run,9853.8,2927.0,1.5250,15027.045


In [15]:
df_athletes_stats.head()
# df_athletes_stats.shape

,athlete_id,Total_distance_run,Total_distance_ride,Total_distance_swim,FC Max,5km PB,10km PB,21km PB,42km PB,Nb_activities,Nb_activities_run,Nb_activities_ride,Nb_activities_swim,Nb_activities_workout,VDOT_5km,VDOT_10km,VDOT_21km,VDOT_42km,VDOT_max
0,44410997,3566056.0,4975555.3,62286.9,200.0,17.92,38.32,83.80,NaN,743.0,541,150.0,38.0,14.0,56.61,54.62,55.36,NaN,56.61
1,45632458,3598572.6,20904903.8,0.0,191.9,23.40,47.10,103.35,245.25,669.0,318,351.0,0.0,0.0,41.42,42.91,43.44,36.92,43.44
2,111468957,3458728.9,2013697.7,39343.2,204.2,19.40,41.15,NaN,NaN,170.0,142,28.0,0.0,0.0,51.61,50.25,NaN,NaN,51.61
3,118945026,3383005.5,1141483.4,62960.8,198.6,17.95,37.15,86.37,191.43,540.0,372,69.0,43.0,56.0,56.50,56.64,53.47,49.77,56.64
4,146587351,1941981.7,0.0,4900.0,203.8,19.90,43.70,109.25,NaN,500.0,329,103.0,64.0,4.0,50.10,46.84,40.71,NaN,50.10


In [16]:
df_run = df_final_by_activity[df_final_by_activity['sport_type'] == 'Run'] # We divide into 4 datasets, for each type of activity
df_bike = df_final_by_activity[df_final_by_activity['sport_type'] == 'Ride']
df_swim = df_final_by_activity[df_final_by_activity['sport_type'] == 'Swim']
df_workout = df_final_by_activity[df_final_by_activity['sport_type'] == 'Workout']

In [17]:
df_run = df_run.drop(columns = ['sport_type'])
df_bike = df_bike.drop(columns = ['sport_type'])
df_swim = df_swim.drop(columns = ['sport_type'])
df_workout = df_workout.drop(columns = ['sport_type'])

In [18]:
total_distance = df_final_by_activity.groupby('athlete_id')['distance'].sum()
total_distance = pd.DataFrame(total_distance).reset_index()
total_distance = total_distance.rename(columns = {0 : 'total_distance'})
total_distance

,athlete_id,distance
0,44410997,10141887.3
1,45632458,25701163.2
2,59727232,2180578.7
3,111468957,5570219.3
4,118945026,4713814.1
5,122317319,35826.9
6,146587351,1961955.4


In [19]:
unique_counts = df_final_by_activity.nunique().sort_values(ascending=False) # Counts the number of unique values in each column
df_unique = pd.DataFrame({'Colonne': unique_counts.index, 'Valeurs Uniques': unique_counts.values})
print(df_unique)

          Colonne  Valeurs Uniques
0     activity_id             3163
1      start_date             3144
2   training_load             2730
3        distance             2690
4     moving_time             2624
5        cv_speed             2168
6       intensity              377
7          pct_Z1              309
8          pct_Z2              308
9          pct_Z3               56
10         pct_Z4               19
11     sport_type               16
12     athlete_id                7


In [20]:
missing_values_count = df_run.isnull().sum()
missing_values_count.sort_values(ascending=False)

athlete_id       0
activity_id      0
start_date       0
cv_speed         0
pct_Z1           0
pct_Z2           0
pct_Z3           0
pct_Z4           0
distance         0
moving_time      0
intensity        0
training_load    0
dtype: int64

In [22]:
df_values = df_best_efforts[df_best_efforts['athlete_id'] == 118945026]["best_effort_name"].value_counts().reset_index() # Counts the number of occurrences for each value of a certain variable
df_values.columns = ["Value", "Occurrences"]
df_values.sort_values(by='Value', ascending = False).head(10)

,Value,Occurrences
5,Marathon,3
2,Half-Marathon,7
0,5K,12
3,30K,4
4,20K,4
1,10K,10


## #1 Running activities analysis ###

### Univariate analysis (activities)

In [ ]:
df_run.dtypes

In [ ]:
df_run.shape

In [ ]:
unique_counts_run = df_run.nunique().sort_values(ascending=False)
df_unique_run = pd.DataFrame({'Column': unique_counts_run.index, 'Unique values': unique_counts_run.values})
print(df_unique_run)

In [ ]:
numerical_var = df_run[['average_heartrate', 'distance', 'moving_time', 'average_speed_km_h']]

fig = plt.figure(figsize=(18, 16))

for index, col in enumerate(numerical_var.columns, 1):
    plt.subplot(3, 2, index)
    sns.histplot(df_run[col].dropna(), kde=False, bins=50)
    plt.title(col)

fig.tight_layout(pad=1.0)
plt.show()

In [ ]:
# categorical_var = df_run.select_dtypes(exclude=['number'])
# categorical_var = categorical_var.drop(columns=['start_date'])
# fig = plt.figure(figsize=(18, 16))

# for index, col in enumerate(categorical_var.columns, 1):
#     plt.subplot(6, 4, index)
#     sns.countplot(df_run[col])
#     plt.xticks(rotation = 90)
#     plt.title(col)

# fig.tight_layout(pad=1.0)
# plt.show()

### Univariate analysis (athletes_stats)

In [ ]:
numerical_var2 = df_athletes_stats[['Total_distance_run', 'Total_distance_ride', 'Total_distance_swim', 'VDOT_5km', 'VDOT_10km', 'VDOT_21km', 'VDOT_42km', 'VDOT_max',
                                     'Nb_activities', 'Nb_activities_run', 'Nb_activities_ride', 'Nb_activities_swim', 'Nb_activities_workout']]

fig = plt.figure(figsize=(18, 16))

for index, col in enumerate(numerical_var2.columns, 1):
    plt.subplot(4, 4, index)
    sns.histplot(df_athletes_stats[col].dropna(), kde=False, bins=50)
    plt.title(col)

fig.tight_layout(pad=1.0)
plt.show()

### Bivariate analysis

In [ ]:
plt.figure(figsize=(10,6))
correlation = numerical_var2.corr()
sns.heatmap(correlation, linewidths=0.5, cmap='Blues', annot=True)

### Clustering

In [ ]:
# Here we will try to do clustering on all athlete sessions: differentiate easy run, threshold, interval training etc...
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

In [ ]:
X = df_run[['activity_id', 'cv_speed', 'pct_Z1', 'pct_Z2', 'pct_Z3', 'pct_Z4', 'intensity', 'training_load', 'distance']]
X = X.drop_duplicates(subset='activity_id')
X = X.drop(columns=['activity_id'])

X_scaled = StandardScaler().fit_transform(X) # Important to normalize the data because units are different

In [ ]:
inertias = []
for i in range(1, 11): # Elbow method to find k
    kmeans = KMeans(n_clusters=i, random_state=0)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)
plt.plot(range(1, 11), inertias)
plt.title('Méthode du coude')
plt.xlabel('Nombre de clusters')
plt.ylabel('Inertie')
plt.show()

In [ ]:
silhouette_scores = []
for i in range(2, 11):  # We also test with the silhouette method (we look for the best score)
    kmeans = KMeans(n_clusters=i, random_state=0)
    labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)

plt.plot(range(2, 11), silhouette_scores)
plt.title('Silhouette score')
plt.xlabel('Number of clusters')
plt.ylabel('Silhouette score')
plt.show()

### Cluster interpretation

In [ ]:
# K=5 seems the best in this case
kmeans = KMeans(n_clusters=3, random_state=0)
labels = kmeans.fit_predict(X_scaled)

In [ ]:
X['cluster'] = kmeans.labels_
cluster_summary = X.groupby('cluster').mean()
display(cluster_summary)

In [ ]:
df_clusters = df_run[['activity_id', 'athlete_id']].drop_duplicates()
df_clusters['cluster'] = kmeans.labels_

# We group by athlete to see the proportions of his activities in each cluster
df_clusters.groupby('athlete_id')['cluster'].value_counts(normalize=True).unstack().fillna(0)

In [ ]:
X['cluster'].value_counts() # Number of activities per cluster

In [ ]:
from sklearn.decomposition import PCA
import seaborn as sns

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
df_plot = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
df_plot['cluster'] = kmeans.labels_

sns.scatterplot(data=df_plot, x='PC1', y='PC2', hue='cluster', palette='tab10')

In [ ]:
X.sort_values(by='training_load').tail()

In [ ]:
df_run['cluster'] = df_clusters['cluster']

In [ ]:
# df_run.tail(10)
df_run.dtypes

In [ ]:
df_athletes_stats.dtypes

In [ ]:
df_run = df_run.sort_values(by=["athlete_id", "start_date"]).copy()

# Time in each zone (in seconds)
df_run["time_Z1"] = df_run["pct_Z1"] * df_run["moving_time"]
df_run["time_Z2"] = df_run["pct_Z2"] * df_run["moving_time"]
df_run["time_Z3"] = df_run["pct_Z3"] * df_run["moving_time"]
df_run["time_Z4"] = df_run["pct_Z4"] * df_run["moving_time"]

# Cumulative time per athlete
df_run["cumulative_time"] = df_run.groupby("athlete_id")["moving_time"].cumsum()
df_run["cumulative_Z1"] = df_run.groupby("athlete_id")["time_Z1"].cumsum()
df_run["cumulative_Z2"] = df_run.groupby("athlete_id")["time_Z2"].cumsum()
df_run["cumulative_Z3"] = df_run.groupby("athlete_id")["time_Z3"].cumsum()
df_run["cumulative_Z4"] = df_run.groupby("athlete_id")["time_Z4"].cumsum()

# Cumulative percentage of time spent in each zone
df_run["pct_time_Z1"] = (df_run["cumulative_Z1"] / df_run["cumulative_time"] * 100).round(2)
df_run["pct_time_Z2"] = (df_run["cumulative_Z2"] / df_run["cumulative_time"] * 100).round(2)
df_run["pct_time_Z3"] = (df_run["cumulative_Z3"] / df_run["cumulative_time"] * 100).round(2)
df_run["pct_time_Z4"] = (df_run["cumulative_Z4"] / df_run["cumulative_time"] * 100).round(2)

In [ ]:
df_run["start_date"] = pd.to_datetime(df_run["start_date"])
df_run = df_run.sort_values(by=["athlete_id", "start_date"])
df_run = df_run.set_index("start_date")


df_run['cumulative_training_load_2_weeks'] = (
    df_run.groupby('athlete_id')['training_load']
    .rolling('14D', min_periods=1)
    .sum()
    .reset_index(level=0, drop=True)
)

df_run['cumulative_training_load_4_weeks'] = (
    df_run.groupby('athlete_id')['training_load']
    .rolling('28D', min_periods=1)
    .sum()
    .reset_index(level=0, drop=True)
)

df_run['cumulative_training_load_8_weeks'] = (
    df_run.groupby('athlete_id')['training_load']
    .rolling('56D', min_periods=1)
    .sum()
    .reset_index(level=0, drop=True)
)
df_run.reset_index(inplace=True)

In [ ]:
def plot_variable_for_athlete(df, athlete_id, variable, title=None, ylabel=None):
    athlete_data = df[df["athlete_id"] == athlete_id]

    plt.figure(figsize=(12, 5))
    plt.plot(athlete_data.index, athlete_data[variable], marker='o', linestyle='-')
    plt.title(title or f"{variable} over time for athlete {athlete_id}")
    plt.xlabel("Date")
    plt.ylabel(ylabel or variable)
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
plot_variable_for_athlete(df=df_run,athlete_id="118945026",variable="pct_time_Z1",title="cumulated training load over 4 weeks",ylabel="Training Load")